[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/13-cost-latency.ipynb)

## Colab setup

On Google Colab, use the setup in `tutorial/common/colab.py`. A fresh Colab runtime does not have this repository, so clone the repo into the session first. Then, before the lesson cells, run `setup_colab` from that module (import it from `tutorial.common.colab` and call it, or open the file and run `setup_colab` from there). The helper installs the tutorial packages, checks the repo out under `/content/book-agents` when it is still missing, and copies a Colab secret named `OPENAI_API_KEY`. Local Jupyter and VS Code can skip it when you already have the repo.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) before you run `setup_colab`. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

# 13. Cost and latency

Every model call logs tokens, latency, and an illustrative cost. Read-only tools are cached. A customer asks for pickup hours and asks for the hours topic twice. The second lookup should print `cache: HIT`. A third identical call would stop the loop.

`route_task` prints whether the question looks like a short lookup or a guided purchase. Both routes name `gpt-4.1-mini` here. Spans from tutorial 12 stay on, because that module is imported.

## Setup

Find the repo so `import tutorial` works, then load the key. `chat` calls the OpenAI Chat Completions API. The key is not printed.

In [ ]:
import sys
from pathlib import Path


def find_root():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "tutorial" / "common" / "client.py").is_file():
            return folder
    raise RuntimeError("Open this notebook inside the book-agents repository.")


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.runtime import require_key

_, model = require_key()
print("model:", model)

## Ledger and cache

The rates in `tutorial.common.cost` are illustrative. They are not an invoice.

In [ ]:
import json

import tutorial.common.facts
import tutorial.common.trace
import tutorial.common.cost  # ledger, route, and read cache

from tutorial.common.cost import CACHE_EVENTS, cost_summary, reset_cache
from tutorial.common.harness import check, system_text
from tutorial.common.loop import run_agent

reset_cache()
question = (
    "What are your pickup hours? Look up the hours topic, "
    "then look that same topic up one more time, and then tell me the hours."
)
result = run_agent(question, system_text(), trace_id="hours-cached")
print("ANSWER:", result["text"])
summary = cost_summary(result["usage"])
print("ledger:", json.dumps(summary))
print("cache events:", CACHE_EVENTS)
check(result["stopped"] == "final", "the hours question finished")
check(any(event.startswith("MISS") for event in CACHE_EVENTS), "the first read missed the cache")
check(any(event.startswith("HIT") for event in CACHE_EVENTS), "the second read hit the cache")
check(summary["prompt_tokens"] > 0, "the ledger counted prompt tokens")
check(summary["estimated_usd"] >= 0, "the ledger has an illustrative cost")
check(len(result["spans"]) >= 2, "spans from tutorial 12 are still recorded")